# Compare GPU Roofline Results

This notebook reloads and re-parses Nsight Compute report files for multiple GPUs, normalizes the data, and compares per-kernel roofline metrics (traffic, FLOPs, arithmetic intensity, and timing) across devices for both CUDA and OpenMP builds.

In [1]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path

from sass_helper import *

/Users/gbolet/miniconda3/envs/gpuflopbench-updated/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + ['sample']

# markers we should ignore / drop kernels containing these from the dataset
#library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [26]:
df = pd.read_csv('all-NCU-GPU-Data.csv', low_memory=False)
print(df.shape)
print(df.columns)


# let's rename the device column
def rename_devices(x):
    if '3080' in x:
        return '3080'
    elif 'A100' in x:
        return 'A100'
    elif 'A10' in x:
        return 'A10'
    elif 'H100' in x:
        return 'H100'
    else:
        raise ValueError(f'Unknown device name in {x}')

df['device'] = df['device'].apply(lambda x: rename_devices(x))


(20044, 2659)
Index(['ID', 'Process ID', 'Process Name', 'Host Name', 'Kernel Name',
       'Context', 'Stream', 'Block Size', 'Grid Size', 'Device',
       ...
       'smsp__sass_inst_executed_op_shared.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_shared_gmma.sum',
       'smsp__sass_inst_executed_op_shared_gmma.sum.pct_of_peak_sustained_elapsed',
       'smsp__sass_inst_executed_op_tma_ld.sum',
       'smsp__sass_inst_executed_op_tma_red.sum',
       'smsp__sass_inst_executed_op_tma_st.sum', 'launch_cluster_dim_x',
       'launch_cluster_dim_z', 'launch_cluster_max_active',
       'launch_cluster_dim_y'],
      dtype='object', length=2659)


In [4]:
sass = pd.read_csv('all-NCU-SASS-Data.csv', low_memory=False)
print(sass.shape)
print(sass.columns)
print(sass.columns[68], sass.columns[74])

(16606832, 76)
Index(['Address', 'Source', 'Warp Stall Sampling (All Samples)',
       'Warp Stall Sampling (Not-issued Samples)', '# Samples',
       'Instructions Executed', 'Thread Instructions Executed',
       'Predicated-On Thread Instructions Executed', 'Avg. Threads Executed',
       'Avg. Predicated-On Threads Executed',
       'Derivative Avg. Predicated-On Threads Executed',
       'Avg. Predicated-On Threads Unexecuted', 'Divergent Branches',
       'Address Space', 'Access Operation', 'Access Size',
       'L1 Tag Requests Global', 'L1 Conflicts Shared N-Way',
       'L1 Wavefronts Shared Excessive', 'L1 Wavefronts Shared',
       'L1 Wavefronts Shared Ideal', 'L2 Theoretical Sectors Global Excessive',
       'L2 Theoretical Sectors Global', 'L2 Theoretical Sectors Global Ideal',
       'L2 Explicit Evict Policies', 'L2 Explicit Hit Policy Evict First',
       'L2 Explicit Hit Policy Evict Last',
       'L2 Explicit Hit Policy Evict Normal',
       'L2 Explicit Hit Policy 

In [5]:
display(sass.head())

,Address,Source,Warp Stall Sampling (All Samples),Warp Stall Sampling (Not-issued Samples),# Samples,Instructions Executed,Thread Instructions Executed,Predicated-On Thread Instructions Executed,Avg. Threads Executed,Avg. Predicated-On Threads Executed,...,stall_wait (Not Issued),Kernel Name,device,codename,source,sample,model_type,Demangled Name,exeArgs,L2 Theoretical Sectors Local
0,0x7563ef27bb00,"MOV R1, c[0x0][0x28]",3,2,3.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
1,0x7563ef27bb10,"S2R R6, SR_CTAID.X",0,0,0.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
2,0x7563ef27bb20,"ULDC.64 UR4, c[0x0][0x118]",48,48,48.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
3,0x7563ef27bb30,"ISETP.GE.AND P0, PT, R6, c[0x0][0x160], PT",2,1,2.0,16384.0,524288.0,524288.0,32.0,32.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN
4,0x7563ef27bb40,"@P0 MOV R0, RZ",2,0,2.0,16384.0,524288.0,0.0,32.0,0.0,...,0.0,_Z15accuracy_kerneliiiPKfPKiPi,3080,accuracy,accuracy-cuda,1,cuda,"accuracy_kernel(int, int, int, const float *, ...",8192 10000 10 100,NaN


In [6]:
sass_columns_to_keep = [
    'Kernel Name',
    'Address',
    'device',
    'codename',
    'exeArgs',
    'source',
    'Source',
    'sample',
    'model_type',
    'Demangled Name',
]
sass = sass[sass_columns_to_keep]

In [7]:
sass = sass.sort_values(by = ['codename', 'Kernel Name', 'device', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], ignore_index=True)

# for each column, we need to make sure it's a string and strip it
for col in sass_columns_to_keep:
    sass[col] = sass[col].astype(str).str.strip()

In [8]:
# this elusive geglu kernel, let's see how many particualar samples of particular addresses we have
# for some strange reason, there is a whole sample that is repeated
subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

# for this subset, display which columns' are different



display(subset)

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
4166904,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167376,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167848,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168320,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168792,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4169264,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."


In [9]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ac200'))]

display(subset)

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
4166616,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167088,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",1,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4167560,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168032,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",2,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168504,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."
4168976,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",3,cuda,"void geglu_kernel<float, float, (int)160, (int..."


In [10]:

display(sass.head(15))

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name
0,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb00,3080,accuracy,8192 10000 10 100,accuracy-cuda,"MOV R1, c[0x0][0x28]",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
1,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb10,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R6, SR_CTAID.X",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
2,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb20,3080,accuracy,8192 10000 10 100,accuracy-cuda,"ULDC.64 UR4, c[0x0][0x118]",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
3,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb30,3080,accuracy,8192 10000 10 100,accuracy-cuda,"ISETP.GE.AND P0, PT, R6, c[0x0][0x160], PT",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
4,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb40,3080,accuracy,8192 10000 10 100,accuracy-cuda,"@P0 MOV R0, RZ",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
5,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb50,3080,accuracy,8192 10000 10 100,accuracy-cuda,@P0 BRA 0x7563ef27bf00,1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
6,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb60,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R11, SR_TID.X",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
7,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb70,3080,accuracy,8192 10000 10 100,accuracy-cuda,"S2R R13, SR_LANEID",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
8,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb80,3080,accuracy,8192 10000 10 100,accuracy-cuda,"SHF.R.S32.HI R0, RZ, 0x1f, R11",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."
9,_Z15accuracy_kerneliiiPKfPKiPi,0x7563ef27bb90,3080,accuracy,8192 10000 10 100,accuracy-cuda,"LEA.HI R2, R0, R11, RZ, 0x5",1,cuda,"accuracy_kernel(int, int, int, const float *, ..."


In [11]:
# for the sass dataframe, group it by 'Kernel Name', 'device', 'codename', 'exeArgs', and 'source'

# for some strange reason, there are duplicate rows, so we need to drop them first
sass['opcode'] = sass['Source'].apply(lambda x: extract_opcode_from_line(x))
sass['is_predicated'] = sass['Source'].apply(lambda x: detect_guard_pred_instruction(x))

print(sass.shape)
sass = sass.drop_duplicates(ignore_index=True).reset_index(drop=True)
print(sass.shape)

sass_grouped = sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'sample', 'model_type', 'Demangled Name'], dropna=False)


(16606832, 12)
(10574472, 12)


In [12]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['device'] == 'H100') &
              (sass['Address'].str.contains('ad400'))]

display(subset)

for col in subset.columns:
    print(subset[col].value_counts())

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2783680,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True
2784152,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",2,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True
2784624,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ad400,H100,geglu,100,geglu-cuda,"@P0 LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",LOP3,True


Kernel Name
_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_    3
Name: count, dtype: int64
Address
0x7e301f7ad400    1
0x7bf83f7ad400    1
0x7e13db7ad400    1
Name: count, dtype: int64
device
H100    3
Name: count, dtype: int64
codename
geglu    3
Name: count, dtype: int64
exeArgs
100    3
Name: count, dtype: int64
source
geglu-cuda    3
Name: count, dtype: int64
Source
@P0   LOP3.LUT R6, R32, 0x80000000, R5, 0xb8, !PT    3
Name: count, dtype: int64
sample
1    1
2    1
3    1
Name: count, dtype: int64
model_type
cuda    3
Name: count, dtype: int64
Demangled Name
void geglu_kernel<float, float, (int)160, (int)1280, (int)8, (int)2>(T1 *, const T1 *)    3
Name: count, dtype: int64
opcode
LOP3    3
Name: count, dtype: int64
is_predicated
True    3
Name: count, dtype: int64


In [13]:

new_sasses = []

counter = 0
# for each group, let's analyze the SASS source code and create an instruction mix profile
# each group should be one particular kernel, we have 3 samples, so the output should have 3 rows per kernel
for name, group in tqdm(sass_grouped):
    group = group.sort_values(by='Address') 

    #print(group.head())
    if name[0] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_' and name[1] == 'H100':
        counter += 1
        print('found a geglu', counter)
        # print the first few instructions
        display(group.head(5))
        display(group.tail(5))

    # create a histogram of the opcodes
    opcode_counts = group['opcode'].value_counts().to_dict()

    # prefix each key with 'opcode_'
    opcode_counts = {f'opcode_{k}': v for k, v in opcode_counts.items()}

    lines_of_code = group.shape[0]

    new_row = {
        'Kernel Name': name[0],
        'device': name[1],
        'codename': name[2],
        'exeArgs': name[3],
        'source': name[4],
        'sample': name[5],
        'model_type': name[6],
        'Demangled Name': name[7],
        'lines_of_code': lines_of_code,
        **opcode_counts
    }
    new_sasses.append(new_row)

    #print(new_row)


new_sass = pd.DataFrame(new_sasses)

# print(sass_grouped.head())

  5%|▌         | 1176/22184 [00:05<00:59, 353.96it/s]

found a geglu 1


,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2783392,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",LDC,False
2783393,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac210,H100,geglu,100,geglu-cuda,"S2R R40, SR_TID.X",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",S2R,False
2783394,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac220,H100,geglu,100,geglu-cuda,"ULDC.64 UR4, c[0x0][0x218]",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",ULDC,False
2783395,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac230,H100,geglu,100,geglu-cuda,"S2R R14, SR_CTAID.X",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",S2R,False
2783396,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7ac240,H100,geglu,100,geglu-cuda,"HFMA2.MMA R13, -RZ, RZ, 0.8349609375, 5.424022...",1,cuda,"void geglu_kernel<float, float, (int)160, (int...",HFMA2,False


,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2783859,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7adf30,H100,geglu,100,geglu-cuda,NOP,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2783860,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7adf40,H100,geglu,100,geglu-cuda,NOP,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2783861,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7adf50,H100,geglu,100,geglu-cuda,NOP,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2783862,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7adf60,H100,geglu,100,geglu-cuda,NOP,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2783863,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e301f7adf70,H100,geglu,100,geglu-cuda,NOP,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False


found a geglu 2


...

,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2784331,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7adf30,H100,geglu,100,geglu-cuda,NOP,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784332,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7adf40,H100,geglu,100,geglu-cuda,NOP,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784333,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7adf50,H100,geglu,100,geglu-cuda,NOP,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784334,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7adf60,H100,geglu,100,geglu-cuda,NOP,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784335,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7bf83f7adf70,H100,geglu,100,geglu-cuda,NOP,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False


found a geglu 3


,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2784336,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac200,H100,geglu,100,geglu-cuda,"LDC R1, c[0x0][0x28]",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",LDC,False
2784337,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac210,H100,geglu,100,geglu-cuda,"S2R R40, SR_TID.X",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",S2R,False
2784338,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac220,H100,geglu,100,geglu-cuda,"ULDC.64 UR4, c[0x0][0x218]",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",ULDC,False
2784339,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac230,H100,geglu,100,geglu-cuda,"S2R R14, SR_CTAID.X",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",S2R,False
2784340,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7ac240,H100,geglu,100,geglu-cuda,"HFMA2.MMA R13, -RZ, RZ, 0.8349609375, 5.424022...",3,cuda,"void geglu_kernel<float, float, (int)160, (int...",HFMA2,False


,Kernel Name,Address,device,codename,exeArgs,source,Source,sample,model_type,Demangled Name,opcode,is_predicated
2784803,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7adf30,H100,geglu,100,geglu-cuda,NOP,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784804,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7adf40,H100,geglu,100,geglu-cuda,NOP,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784805,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7adf50,H100,geglu,100,geglu-cuda,NOP,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784806,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7adf60,H100,geglu,100,geglu-cuda,NOP,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False
2784807,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,0x7e13db7adf70,H100,geglu,100,geglu-cuda,NOP,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",NOP,False


100%|██████████| 22184/22184 [00:10<00:00, 2190.98it/s]


In [14]:
print(new_sass.head())

                                 Kernel Name device      codename    exeArgs  \
0  _Z10QTC_devicePfPcS0_PiS1_S1_S_S1_iiifiii   3080  qtclustering  --Verbose   
1  _Z10QTC_devicePfPcS0_PiS1_S1_S_S1_iiifiii   3080  qtclustering  --Verbose   
2  _Z10QTC_devicePfPcS0_PiS1_S1_S_S1_iiifiii   3080  qtclustering  --Verbose   
3  _Z10QTC_devicePfPcS0_PiS1_S1_S_S1_iiifiii    A10  qtclustering  --Verbose   
4  _Z10QTC_devicePfPcS0_PiS1_S1_S_S1_iiifiii    A10  qtclustering  --Verbose   

              source sample model_type  \
0  qtclustering-cuda      1       cuda   
1  qtclustering-cuda      2       cuda   
2  qtclustering-cuda      3       cuda   
3  qtclustering-cuda      1       cuda   
4  qtclustering-cuda      2       cuda   

                                      Demangled Name  lines_of_code  \
0  QTC_device(float *, char *, char *, int *, int...           1480   
1  QTC_device(float *, char *, char *, int *, int...           1480   
2  QTC_device(float *, char *, char *, int *, int... 

In [15]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

subset = new_sass[(new_sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_')]

display(subset)


,Kernel Name,device,codename,exeArgs,source,sample,model_type,Demangled Name,lines_of_code,opcode_IMAD,opcode_MOV,opcode_BRA,opcode_ISETP,opcode_IADD3,opcode_LDG,opcode_LEA,opcode_FSETP,opcode_BSSY,opcode_BSYNC,opcode_SHF,opcode_LDS,opcode_STG,opcode_SEL,opcode_BREAK,opcode_NOP,opcode_FSEL,opcode_STS,opcode_BAR,opcode_S2R,opcode_PRMT,opcode_EXIT,opcode_LOP3,opcode_FMUL,opcode_IMNMX,opcode_CALL,opcode_ULDC,opcode_FADD,opcode_HFMA2,opcode_LDC,opcode_UMOV,opcode_UIADD3,opcode_S2UR,opcode_ULEA,opcode_UIMAD,opcode_R2UR,opcode_UISETP,opcode_USHF,opcode_VIADD,opcode_USEL,opcode_PLOP3,opcode_FFMA,opcode_F2I,opcode_FCHK,opcode_FRND,opcode_P2R,opcode_I2FP,opcode_I2F,opcode_ULOP3,opcode_YIELD,opcode_WARPSYNC,opcode_DADD,opcode_MUFU,opcode_VIADDMNMX,opcode_SHFL,opcode_ATOMG,opcode_VOTE,opcode_ENDCOLLECTIVE,opcode_DFMA,opcode_DMUL,opcode_CS2R,opcode_ATOMS,opcode_REDUX,opcode_UFLO,opcode_VOTEU,opcode_IABS,opcode_HMMA,opcode_DSETP,opcode_R2P,opcode_F2F,opcode_LD,opcode_ST,opcode_LDL,opcode_STL,opcode_BMOV,opcode_RET,opcode_VIMNMX,opcode_FMNMX,opcode_VIMNMX3,opcode_UP2UR,opcode_POPC,opcode_UPRMT,opcode_FLO,opcode_FSET,opcode_HADD2,opcode_F2FP,opcode_HMUL2,opcode_HSET2,opcode_BRX,opcode_DEPBAR,opcode_VABSDIFF4,opcode_BREV,opcode_UBREV,opcode_HMNMX2,opcode_HSETP2,opcode_UPLOP3,opcode_CCTL,opcode_ERRBAR,opcode_MEMBAR,opcode_CGAERRBAR,opcode_B2R,opcode_UPOPC,opcode_SGXT,opcode_MATCH,opcode_ATOM,opcode_QSPC,opcode_NANOSLEEP,opcode_USGXT,opcode_BRXU,opcode_F2IP
1551,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,3080,geglu,100,geglu-cuda,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",464,2.0,8.0,1,NaN,NaN,8.0,4.0,16.0,NaN,NaN,4.0,NaN,4.0,NaN,NaN,13,144.0,NaN,NaN,2.0,NaN,1.0,16.0,80.0,NaN,NaN,1.0,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1552,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,3080,geglu,100,geglu-cuda,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",464,2.0,8.0,1,NaN,NaN,8.0,4.0,16.0,NaN,NaN,4.0,NaN,4.0,NaN,NaN,13,144.0,NaN,NaN,2.0,NaN,1.0,16.0,80.0,NaN,NaN,1.0,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1553,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,3080,geglu,100,geglu-cuda,3,cuda,"void geglu_kernel<float, float, (int)160, (int...",464,2.0,8.0,1,NaN,NaN,8.0,4.0,16.0,NaN,NaN,4.0,NaN,4.0,NaN,NaN,13,144.0,NaN,NaN,2.0,NaN,1.0,16.0,80.0,NaN,NaN,1.0,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1554,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,A10,geglu,100,geglu-cuda,1,cuda,"void geglu_kernel<float, float, (int)160, (int...",464,2.0,8.0,1,NaN,NaN,8.0,4.0,16.0,NaN,NaN,4.0,NaN,4.0,NaN,NaN,13,144.0,NaN,NaN,2.0,NaN,1.0,16.0,80.0,NaN,NaN,1.0,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1555,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,A10,geglu,100,geglu-cuda,2,cuda,"void geglu_kernel<float, float, (int)160, (int...",464,2.0,8.0,1,NaN,NaN,8.0,4.0,16.0,NaN

In [16]:
# sanity check, these opcodes should be identical across samples of the same kernel
new_sass_grouped = new_sass.groupby(['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name'], dropna=False)

condensed_imix = []

for name, group in tqdm(new_sass_grouped):
    if group.shape[0] > 1:
        group = group.sort_values(by='sample')

        mismatch_found = False

        # if the lines_of_code are different, then we have a problem
        first_loc = group.iloc[0]['lines_of_code']
        for idx, row in group.iterrows():
            if row['lines_of_code'] != first_loc:
                print("Lines of code mismatch found in group:", name)
                #display(group)
                mismatch_found = True
                break

        if mismatch_found:
            continue

    condensed_imix.append(group.iloc[0])

        # this still works, but is too verbose
        # check if all rows are identical
        #first_row = group.iloc[0]
        #first_row = first_row.drop('sample')
        #for idx, row in group.iterrows():
        #    # drop the sample index column for comparison
        #    if row.sample:
        #        row = row.drop('sample')
        #    if not row.equals(first_row):
        #        print("Mismatch found in group:", name)
        #        display(group)
        #        # print the columns that are different
        #        diff_cols = first_row[first_row != row].index.tolist()
        #        print("Different columns:", diff_cols)
        #        # print the values of the different columns
        #        for col in diff_cols:
        #            print(f"Column: {col}, idx: {idx}, Value 1: {first_row[col]}, Value 2: {row[col]}")
            #assert row.equals(first_row), "Mismatch found in group"


condensed_imix_df = pd.DataFrame(condensed_imix)

100%|██████████| 7412/7412 [00:01<00:00, 6309.78it/s]


In [ ]:
# let's fix the omp kernel names 

def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x
# the rest of the chars are the function name and line number -- which is the same across compilations
condensed_imix_df['Kernel Name'] = condensed_imix_df['Kernel Name'].apply(fix_omp_kernel_name)

In [27]:
df['Kernel Name'] = df['Kernel Name'].apply(fix_omp_kernel_name)

In [18]:
print(condensed_imix_df.shape)

print(condensed_imix_df['device'].value_counts())

(7412, 119)
device
3080    1991
H100    1884
A10     1827
A100    1710
Name: count, dtype: int64


In [ ]:
condensed_imix_df.to_csv('all-IMIX-Data.csv', index=False)


In [24]:
print(df.shape)

# let's get overlapping column names
overlap_cols = list(set(df.columns).intersection(set(condensed_imix_df.columns)))
print("Overlapping columns:", overlap_cols)

# lets get lowered overlapping names
overlap_cols = list(set(df.columns.str.lower()).intersection(set(condensed_imix_df.columns.str.lower())))
print("Lowered Overlapping columns:", overlap_cols)

(20044, 2659)
Overlapping columns: ['codename', 'device', 'exeArgs', 'sample', 'Demangled Name', 'Kernel Name', 'model_type', 'source']
Lowered Overlapping columns: ['codename', 'device', 'exeargs', 'sample', 'demangled name', 'kernel name', 'model_type', 'source']


In [28]:
# check device value counts
print(condensed_imix_df['device'].value_counts())
print(df['device'].value_counts())


device
3080    1991
H100    1884
A10     1827
A100    1710
Name: count, dtype: int64
device
3080    5439
H100    5096
A10     4971
A100    4538
Name: count, dtype: int64


In [32]:
merge_cols = ['Kernel Name', 'device', 'codename', 'exeArgs', 'source', 'model_type', 'Demangled Name']
# now let's merge the imix data with the perf counter data
merged_df = pd.merge(df, condensed_imix_df, on=merge_cols, suffixes=('', '_imix'))

In [36]:
print(merged_df.shape)

print(merged_df[merge_cols].head(10))

assert merged_df.columns.str.contains('opcode').any(), "No opcode columns found in merged dataframe"
assert merged_df.columns.str.contains('smsp').any(), "No perf counter columns found in merged dataframe"

(16751, 2771)
                                       Kernel Name device  codename  \
0                   _Z15accuracy_kerneliiiPKfPKiPi   3080  accuracy   
1                   _Z15accuracy_kerneliiiPKfPKiPi   3080  accuracy   
2                   _Z15accuracy_kerneliiiPKfPKiPi   3080  accuracy   
3                                         main_l57   3080  accuracy   
4                                         main_l57   3080  accuracy   
5                                         main_l57   3080  accuracy   
6    _Z14calculateForcePA400_A400_dS1_S1_S1_dddddd   3080       ace   
7  _Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd   3080       ace   
8            _Z21boundaryConditionsPhiPA400_A400_d   3080       ace   
9    _Z15thermalEquationPA400_A400_dS1_S1_S1_ddddd   3080       ace   

             exeArgs         source model_type  \
0  8192 10000 10 100  accuracy-cuda       cuda   
1  8192 10000 10 100  accuracy-cuda       cuda   
2  8192 10000 10 100  accuracy-cuda       cuda   
3  8

In [37]:
merged_df.to_csv('all-NCU-GPU-Data-with-IMIX.csv', index=False)